In [ ]:
import pandas as pd 
import numpy as np 
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
import pandas as pd
from numpy import dot, exp
import random
from sklearn.model_selection import KFold
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.metrics import log_loss
from sklearn.linear_model import LogisticRegression
import copy
import os

#kaggle read csv

In [ ]:
# train = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/train.csv')
# test = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/test.csv')
# greeks = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/greeks.csv')

#local read csv

In [ ]:
filepath = '../data/'
train = pd.read_csv(os.path.join(filepath, 'train.csv'), index_col='Id')
test = pd.read_csv(os.path.join(filepath, 'test.csv'), index_col="Id")
greeks = pd.read_csv(os.path.join(filepath, 'greeks.csv'), index_col="Id")

# quick data peek

In [ ]:
train.head(5)


In [ ]:
test.head(5)


In [ ]:
greeks.head(5)

# greeks used in stratified k-fold strategy
reference: https://www.kaggle.com/code/tatudoug/logistic-regression-baseline

In [ ]:
# sum greeks, put into one column
greeks['k'] = greeks['Alpha']+greeks['Beta']+greeks['Gamma']+greeks['Delta']
greeks = greeks.reset_index()
train = pd.merge(greeks[['k','Id']],train,on='Id')
#set Id as index
train = train.set_index('Id')
#quick check after merge and set index
display(train.head(5))
# name columns
names = ['AB', 'AF', 'AH', 'AM', 'AR', 'AX', 'AY', 'AZ', 'BC', 'BD ', 'BN',
       'BP', 'BQ', 'BR', 'BZ', 'CB', 'CC', 'CD ', 'CF', 'CH', 'CL', 'CR', 'CS',
       'CU', 'CW ', 'DA', 'DE', 'DF', 'DH', 'DI', 'DL', 'DN', 'DU', 'DV', 'DY',
       'EB', 'EE', 'EG', 'EH', 'EJ', 'EL', 'EP', 'EU', 'FC', 'FD ', 'FE', 'FI',
       'FL', 'FR', 'FS', 'GB', 'GE', 'GF', 'GH', 'GI', 'GL']
target_name = 'Class'

# data cleanup

In [ ]:
# encode features
train['EJ'] = pd.Series(np.where(train.EJ.values == 'A', 1, 0),
          train.index)
test['EJ'] = pd.Series(np.where(test.EJ.values == 'A', 1, 0),
          test.index)
# fill nan data with mean values 
train[names] = train[names].fillna(train[names].mean())
test[names] = test[names].fillna(train[names].mean())
# clip values to avoid different values in the test set from train
test = test[names].clip(train[names].min(axis=0).values,train[names].max(axis=0).values, axis=1)
# data scaled to allow the features interaction (by multiplication)
scaler = StandardScaler()
train2 = copy.copy(train)
teste2 = copy.copy(test)
vals = scaler.fit_transform(train[names])
vals_test = scaler.transform(test[names])
train2[names] = vals
teste2[names] = vals_test

#what does the scaled data look like?

In [ ]:
display(train2.head(20))

# Feature Engineering

# 2 order interactions:

mutliplying two together

In [ ]:
def mult(df,nome1,nome2):
    a  = df[nome1]*df[nome2]
    return(a/max(a))

def sqrt_exp(df,n1,n2):
    a = df[n1]**2 + df[n2]**2
    return(a/max(a))

def div(df,nome1,nome2):
    a  = df[nome1]/df[nome2]
    if max(a) == 0:
        print("bad!")
    return(a/max(a))

h = []
ht = []

n = 1
for n1 in names:
    for n2 in names[n:]:
        h.append(mult(train2,n1,n2).rename(n1+'_mul_'+n2))
        h.append(sqrt_exp(train2,n1,n2).rename(n1+'_sqrt_exp_'+n2))
        # h.append(div(train2,n1,n2).rename(n1+'_div_'+n2))

        ht.append(mult(teste2,n1,n2).rename(n1+'_mul_'+n2))
        ht.append(sqrt_exp(teste2,n1,n2).rename(n1+'_sqrt_exp_'+n2))
        # ht.append(div(teste2,n1,n2).rename(n1+'_div_'+n2))

    n+=1
    
newF = pd.DataFrame(h)
newF_test = pd.DataFrame(ht)

display(newF)
display(newF_test)

Information Value (IV) and Weights of evidence (WoE) are widely used in credit score models with logistic regression. IV is a bivariate metric which ranks the predictive power of the the variables in relation to the target. WoE will transform the data to a monotonic pattern which is fundamental to the Logistic Regression model to perform well.

In [ ]:
#https://lucastiagooliveira.github.io/datascience/iv/woe/python/2020/12/15/iv_woe.html
def iv_woe(data, target, bins=10, show_woe=False):
    
    #Empty Dataframe
    newDF,woeDF = pd.DataFrame(), pd.DataFrame()
    
    #Extract Column Names
    cols = data.columns
    
    #Run WOE and IV on all the independent variables
    for ivars in cols[~cols.isin([target])]:
        if (data[ivars].dtype.kind in 'bifc') and (len(np.unique(data[ivars]))>10):
            binned_x = pd.qcut(data[ivars], bins,  duplicates='drop')
            d0 = pd.DataFrame({'x': binned_x, 'y': data[target]})
        else:
            d0 = pd.DataFrame({'x': data[ivars], 'y': data[target]})

        
        # Calculate the number of events in each group (bin)
        d = d0.groupby("x", as_index=False).agg({"y": ["count", "sum"]})
        d.columns = ['Cutoff', 'N', 'Events']
        
        # Calculate % of events in each group.
        d['% of Events'] = np.maximum(d['Events'], 0.5) / d['Events'].sum()

        # Calculate the non events in each group.
        d['Non-Events'] = d['N'] - d['Events']
        # Calculate % of non events in each group.
        d['% of Non-Events'] = np.maximum(d['Non-Events'], 0.5) / d['Non-Events'].sum()

        # Calculate WOE by taking natural log of division of % of non-events and % of events
        d['WoE'] = np.log(d['% of Events']/d['% of Non-Events'])
        d['IV'] = d['WoE'] * (d['% of Events'] - d['% of Non-Events'])
        d.insert(loc=0, column='Variable', value=ivars)
        #print("Information value of " + ivars + " is " + str(round(d['IV'].sum(),6)))
        temp =pd.DataFrame({"Variable" : [ivars], "IV" : [d['IV'].sum()]}, columns = ["Variable", "IV"])
        newDF=pd.concat([newDF,temp], axis=0)
        woeDF=pd.concat([woeDF,d], axis=0)

        #Show WOE Table
        if show_woe == True:
            print(d)
    return newDF, woeDF

In [ ]:
a,b = iv_woe(train2, target_name, bins=10, show_woe=False)

In [ ]:
# most important features based on IV
a.sort_values(by='IV',ascending=False).Variable.values 

In [ ]:
# Reordering the dataframe to keep IV with higger values in front
trainE = train[a.sort_values(by='IV',ascending=False).Variable.values]
trainE[target_name] = train[target_name]
testeE = test[a.sort_values(by='IV',ascending=False).Variable.values[2:]]

# join the original vars and the interactions between them
ff = pd.concat([trainE,newF.T],axis=1)
ff_teste = pd.concat([testeE,newF_test.T],axis=1)

a,b = iv_woe(ff, target_name, bins=10, show_woe=False)

# deleting all IVs below 0.05
a = a.loc[a['IV']> 0.05]

allNames = a.sort_values(by='IV',ascending=False).Variable.values
crossNames = [x for x in allNames if '_mul_' in x]
logCrossNames = [x for x in allNames if '_log_mul_' in x]

nomes2 = list(trainE)+crossNames+logCrossNames
nomes2.remove('Class')

In [ ]:
# threshold to correlation features
threshold = 0.45

cc = ff[nomes2[2:]].corr()

mat_x = abs(cc)>threshold
mat_x = mat_x.to_numpy()

In [ ]:
# selection tof the variables with low correlation, there are +- 70 features with low correlation
var1 = []
nomes = list(cc)
var1.append(nomes[0])
max_vars = 100

count = 1
for n in range(1,len(nomes)):
    
    if (mat_x[n,:n+1].sum() ) == 1:
        
        var1.append(nomes[n])        
        count+=1
        
        if(count == max_vars):
            break

In [ ]:
var1 

In [ ]:
# create dic with WoE transformation
list_dics = []

for var in var1:
  df_temp = b.loc[b['Variable']==var].reset_index()
  # creating dictionary
  dict_var = {}
  for x in range(len(df_temp)):
    line = df_temp.iloc[x]
    dict_var[ line['Cutoff'] ] = line['WoE']
  list_dics.append(dict_var)

In [ ]:
# train and test data
df_original = ff[var1+[target_name]+ ['k'] ]
df_teste2 = ff_teste[var1]
names = var1

In [ ]:
# pandas is giving multiple warnings and making the code "dirty"
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# In this part there is some data leakage as the map is using the full dataset
n= 0

for var in var1:
  df_original.loc[:,var] = df_original[var].map(list_dics[n])
  df_teste2.loc[:,var] = df_teste2[var].map(list_dics[n])
  
  n=n+1

In [ ]:
df_original.loc[:,names] = df_original[names].fillna(df_original[names].mean())
df_teste2.loc[:,names] = df_teste2[names].fillna(df_original[names].mean())

In [ ]:
#https://www.kaggle.com/competitions/icr-identify-age-related-conditions/discussion/409691
from sklearn.utils import class_weight

def balanced_log_loss(y_true, y_pred):
    class_weights = class_weight.compute_class_weight('balanced', classes = np.unique(y_true), y = y_true)
    weights = class_weights[y_true.astype(int)]
    loss = log_loss(y_true, y_pred, sample_weight=weights)
    return loss

In [ ]:
n_splits = 10

predictions_LR = 0
cv_score_LR = 0
train_score_LR = 0

rr = [42,21,100,45,1]
for v_fold in rr:
    print(v_fold)
    skf = StratifiedKFold(n_splits=n_splits,shuffle=True, random_state=v_fold)
    for i, (train_index, test_index) in enumerate(skf.split(df_original[target_name], df_original['k'])):

            model = LogisticRegression(random_state=0,C=0.1,n_jobs=-1,max_iter=2000,class_weight='balanced')

            df_train = df_original.iloc[train_index]
            df_test = df_original.iloc[test_index]
            
            
            df_train1 = df_train[names].to_numpy()
            df_test1 = df_test[names].to_numpy()            
            
            
            model.fit( df_train1, df_train[target_name])

            y_hat_teste_LR = model.predict_proba(df_test1)
            cv_score_LR += balanced_log_loss(df_test[target_name],y_hat_teste_LR)       


            y_hat_train_LR = model.predict_proba(df_train1) 
            train_score_LR += balanced_log_loss(df_train[target_name],y_hat_train_LR)  
            predictions_LR += model.predict_proba(df_teste2[names])
                  

         
print('Train :', train_score_LR/(10*len(rr)) )
print('CV score: ', cv_score_LR/(10*len(rr)) )
predictions_LR= predictions_LR/(10*len(rr))


display(predictions_LR)

In [ ]:
print(train.head(5))
train = train.reset_index()
id_df = pd.DataFrame(train['Id'])
pred_df = pd.DataFrame(predictions_LR)


submission = pd.DataFrame(
    data=id_df.join(pred_df),
    columns=['Id','class_0', 'class_1'],
)
#submission['Id'] = train['Id']
print(submission)

In [ ]:
submission.to_csv('/kaggle/working/submission.csv',index=None)